In [110]:
import os
import torch
import pandas as pd
from torch.utils import data 
from torch import nn
batch_size=64
lr=0.1
k_fold=5
epoch=150
weight_decay=0.001

In [111]:
# read dataset
data_dir="/kaggle/input/competitions/house-prices-advanced-regression-techniques"
train_path=os.path.join(data_dir,'train.csv')
test_path=os.path.join(data_dir,'test.csv')
train_data=pd.read_csv(train_path)
test_data=pd.read_csv(test_path)
train_data.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [112]:
# qie pian
train_features=train_data.iloc[:,1:-1]
train_labels=train_data.iloc[:,-1]
test_features=test_data.iloc[:,1::]
print(train_features.shape,train_labels.shape,test_features.shape)
type(train_labels)



(1460, 79) (1460,) (1459, 79)


pandas.core.series.Series

In [113]:
# merge train_data and test_data to clean data
all_features=pd.concat([train_features,test_features],axis=0)
print(all_features.shape)
print(train_features.dtypes)

(2919, 79)
MSSubClass         int64
MSZoning          object
LotFrontage      float64
LotArea            int64
Street            object
                  ...   
MiscVal            int64
MoSold             int64
YrSold             int64
SaleType          object
SaleCondition     object
Length: 79, dtype: object


In [114]:
# biao zhun hua
num_columns=all_features.select_dtypes(include=['number']).columns
all_features[num_columns] = (
    all_features[num_columns] -
    all_features[num_columns].mean()
) / all_features[num_columns].std()
# tian chong que shi zhi
all_features[num_columns]=(all_features[num_columns].fillna(0))
# one hot
all_features = pd.get_dummies(
    all_features,
    dummy_na=True
)
# trange to float32
all_features = all_features.astype("float32")
#  test and train
num_train=train_features.shape[0]
train_data=all_features.iloc[:num_train]
test_data=all_features.iloc[num_train::]

# trange to tensor
train_features=torch.tensor(train_data.to_numpy(),dtype=torch.float32)
test_features=torch.tensor(test_data.to_numpy(),dtype=torch.float32)
train_labels=torch.tensor(train_labels.to_numpy(),dtype=torch.float32).reshape(-1,1)
print(train_features.size(),test_features.size(),train_labels.size())

torch.Size([1460, 330]) torch.Size([1459, 330]) torch.Size([1460, 1])


In [115]:
def get_net():
    return nn.Sequential(nn.Linear(330,64),nn.ReLU(),nn.Dropout(0.1),nn.Linear(64,1))

loss = nn.MSELoss()

def get_k_fold_data(k,i,train_features,train_labels):
    num=train_features.shape[0]
    begin=i*num//k
    end=(i+1)*num//k
    valid_f=train_features[begin:end]
    valid_l=train_labels[begin:end]
    train_f=torch.cat([train_features[:begin],train_features[end:]],dim=0)
    train_l=torch.cat([train_labels[:begin],train_labels[end:]],dim=0)
    return train_f,train_l,valid_f,valid_l
def data_loader(data_array,batch_size,shuffle1):
    dataset=data.TensorDataset(*(data_array))
    return data.DataLoader(dataset,batch_size,shuffle=shuffle1)

def init(m):
    if type(m)==nn.Linear:
        nn.init.xavier_uniform_(m.weight)
        nn.init.zeros_(m.bias)
def log_rmse(net,x,y):
    with torch.no_grad():
        predic=net(x)
        predic=torch.clamp(predic,min=1)
        result=torch.sqrt(loss(torch.log(predic),torch.log(y)))
        return result.item()
    


In [116]:
for i in range(k_fold):
    train_f,train_l,valid_f,valid_l=get_k_fold_data(k_fold,i,train_features,train_labels)
    train_iter=data_loader((train_f,train_l),batch_size,True)
    valid_iter=data_loader((valid_f,valid_l),batch_size,False)
    net=get_net()
    net.apply(init)

    decay_params=[]
    no_decay_params=[]
    for name,param in net.named_parameters():
        if name.endswith("bias"):
            no_decay_params.append(param)
        else:
            decay_params.append(param)
    trainer=torch.optim.Adam([
        {
            "params":decay_params,
            "weight_decay":weight_decay
        },
        {
            "params":no_decay_params,
            
        }
    ],lr=lr)
    train_score=0.0
    valid_score=0.0
    for j in range(epoch):
        net.train()
        for x,y in train_iter:
            trainer.zero_grad()
            l=loss(net(x),y)
            l.backward()
            trainer.step()
        

    net.eval()
    train_score=log_rmse(net,train_f,train_l)
    valid_score=log_rmse(net,valid_f,valid_l)
    print(f"第{i}折的训练分数{train_score},验证分数{valid_score}")
    
    
    

第0折的训练分数0.1003619134426117,验证分数0.16881580650806427
第1折的训练分数0.09535197913646698,验证分数0.16072826087474823
第2折的训练分数0.11369042843580246,验证分数0.15951059758663177
第3折的训练分数0.09298747777938843,验证分数0.13602691888809204
第4折的训练分数0.08095521479845047,验证分数0.15165118873119354


In [117]:
final_train_iter=data_loader((train_features,train_labels),batch_size,True)
net=get_net()
net.apply(init)

decay_params=[]
no_decay_params=[]
for name,param in net.named_parameters():
    if name.endswith("bias"):
        no_decay_params.append(param)
    else:
        decay_params.append(param)
trainer=torch.optim.Adam([
    {
        "params":decay_params,
        "weight_decay":weight_decay
    },
    {
        "params":no_decay_params,
            
    }
],lr=lr)
for i in range(epoch):
    net.train()
    for x,y in final_train_iter:
        trainer.zero_grad()
        l=loss(net(x),y)
        l.backward()
        trainer.step()
net.eval()
print(f"最终训练集得分：{log_rmse(net,train_features,train_labels):.5f}")

最终训练集得分：0.09849


In [122]:
with torch.no_grad():
    predic=net(test_features)
    predic = torch.clamp(predic, min=1)
    predic=predic.reshape(-1).numpy()
    print(predic.shape)
    print(predic[:10])

(1459,)
[111634.61 135126.02 174930.2  187744.42 197738.48 173056.94 181212.89
 161602.88 192202.25 122329.07]


In [124]:
sample_path = os.path.join(
    data_dir,
    "sample_submission.csv"
)

submission = pd.read_csv(sample_path)

submission["SalePrice"] = predic

submission_path = "/kaggle/working/submission.csv"

submission.to_csv(
    submission_path,
    index=False
)

print("提交文件路径：", submission_path)
print("提交文件形状：", submission.shape)
print("缺失值数量：")
print(submission.isna().sum())

submission.head()

提交文件路径： /kaggle/working/submission.csv
提交文件形状： (1459, 2)
缺失值数量：
Id           0
SalePrice    0
dtype: int64


,Id,SalePrice
0,1461,111634.609375
1,1462,135126.015625
2,1463,174930.203125
3,1464,187744.421875
4,1465,197738.484375


In [125]:
print(submission["SalePrice"].describe())

count      1459.000000
mean     176571.984375
std       78188.320312
min        7996.062500
25%      124856.332031
50%      157711.125000
75%      205831.585938
max      611140.687500
Name: SalePrice, dtype: float64
